In [18]:
import json
import glob
import argparse
import os
import subprocess
import numpy as np
from pathlib import Path

from ar_processor import ARProcessor
from nullsar_tools import parse_cand_file, parse_par_file, parse_JSON, rsync, fit_time_phase, fit_phase_offset, scale_freq_phase



class NullerProcess:
    def __init__(self, tag, processing_args, out_dir, work_dir, mode='INIT'):

        self.out_dir = os.getcwd() if out_dir == 'cwd' else out_dir
        self.work_dir = os.getcwd() if work_dir == 'cwd' else work_dir

        self.processing_args_path = processing_args
        self.processing_args = parse_JSON(processing_args)['nullsar']

        self.tag = tag
        self.mode = mode
        self.processing_dir = f'{self.out_dir}/PROCESSING/{self.tag}'

        self.ar_data = {}

    def injector_setup(self):
        self.check_SNR()

        self.extract_archive()
        self.create_injection_plan()

    def check_SNR(self):
        files_dir = f'/home/rsenzel/Code/inception/dev_nulsar/files'
        SNR_limit = self.processing_args.get('SNR_limit', 15)
        snr_path = f"/home/rsenzel/Code/inception/dev_nulsar/files/FOLDS/outs/INIT_fold_params.json"

        if self.mode == "INIT":
            self.SNR_record = {}
            for par_file  in list(self.processing_args['par_files']):
                psr_ID = Path(par_file).stem
                fits_path = f'{files_dir}/FOLDS/{psr_ID}_mode_INIT.fits'
                archive = ARProcessor(fits_path, mode='load')
                SNR = archive.get_SNR()
                self.SNR_record[psr_ID] = {"SNR": SNR}

            with open(snr_path, 'w') as file:
                json.dump(self.SNR_record, file, indent=4)

    def extract_archive(self):
        par_files = self.processing_args['par_files']
        ar_path = f"/home/rsenzel/Code/inception/dev_nulsar/files/FOLDS/INIT_fold_params.json"
        
        for par_file in par_files:
            psr_ID = Path(par_file).stem
            if self.mode == 'INIT':
                fits_path = f'/home/rsenzel/Code/inception/dev_nulsar/files/FOLDS/{psr_ID}_mode_INIT.fits'
                archive_INIT = ARProcessor(fits_path, mode='load')
                self.parse_archive(psr_ID, archive_INIT, archive_INIT, ar_path)

            if self.mode == 'NULL':
                fits_path_INIT = f'/home/rsenzel/Code/inception/dev_nulsar/files/FOLDS/{psr_ID}_mode_INIT.fits'
                fits_path_OPT = f'/home/rsenzel/Code/inception/dev_nulsar/files/FOLDS/{psr_ID}_mode_OPTIMISE.fits'

                archive_INIT = ARProcessor(fits_path_INIT, mode='load')
                archive_OPT = ARProcessor(fits_path_OPT, mode='load')

                self.parse_archive(psr_ID, archive_INIT, archive_OPT, ar_path)


        if self.mode == "INIT":
            with open(ar_path, 'w') as file:
                for key, value in self.SNR_record.items():
                    if not self.ar_data.get(key, None):
                        self.ar_data[key] = value
                json.dump(self.ar_data, file, indent=4)

    def parse_archive(self, psr_ID, archive_INIT, archive_OPT, ar_path):


        obs_len =  7143.97332819

        profile_path = f"/home/rsenzel/Code/inception/dev_nulsar/files/FOLDS/outs/profile_{psr_ID}.npy"        
        if self.mode == 'INIT':
            SNR = archive_INIT.get_SNR()
            DM = archive_INIT.get_DM()
            freq_phase = archive_INIT.get_freq_phase()
            intensity_profile = archive_INIT.get_intensity_prof()
            
            time_phase = archive_INIT.get_time_phase()
            freq_deriv, phase_offset, SNR_fit = fit_time_phase(time_phase, intensity_profile, obs_len)

            freq_phase_scaled = scale_freq_phase(freq_phase, intensity_profile)
            print(SNR, SNR_fit)
            np.save(profile_path, freq_phase_scaled)

        elif self.mode == "NULL":
            init_ar_data =  parse_JSON(ar_path)
            SNR = init_ar_data[psr_ID]['SNR']
            DM = init_ar_data[psr_ID]['DM']
            freq_deriv = init_ar_data[psr_ID]['FX']

            intensity_profile_INIT = archive_INIT.get_intensity_prof()
            intensity_profile_OPT = archive_OPT.get_intensity_prof()

            phase_offset, SNR_scale = fit_phase_offset(intensity_profile_OPT, intensity_profile_INIT)
            SNR *= SNR_scale
            phase_offset += init_ar_data[psr_ID]['phase_offset']

        self.ar_data[psr_ID] = {"SNR": SNR,  
                                "DM": DM,
                                "phase_offset": phase_offset, 
                                "profile": profile_path,
                                "FX": freq_deriv}
            
    def create_injection_plan(self):
        injection_plan = {
            "psr_global": {
                "injection_id": self.mode,
                "global_seed": 404,
                "create_parfile": 0,
            },

            "pulsars": []
        }

        par_files = self.processing_args['par_files']
        for par_file in par_files:
            psr_ID = Path(par_file).stem
            params = parse_par_file(par_file)

            psr_dict = {
                "ID": psr_ID,
                "mode": "pint",
                "PEPOCH": 0.5,
                "phase_offset": float(self.ar_data[psr_ID]['phase_offset']),
                
                "P0_SNR": 1/float(params['F0']),
                "DM": self.ar_data[psr_ID]['DM'],
                "SNR": -self.ar_data[psr_ID]['SNR'],

                "profile": self.ar_data[psr_ID]['profile'],
                "polycos": par_file
            }

            for key, value in self.ar_data[psr_ID]['FX'].items():
                psr_dict[key] = value

            injection_plan['pulsars'].append(psr_dict)

        self.inject_file = f"{self.work_dir}/NULLSAR_inject_file_mode_{self.mode}.json"
        with open(self.inject_file, 'w') as file:
            json.dump(injection_plan, file, indent=4)

In [19]:
processing_args = '/home/rsenzel/Code/inception/dev_nulsar/files/FOLDS/null_parms.json'
work_dir = '/home/rsenzel/Code/inception/dev_nulsar/files/FOLDS/outs'

In [20]:
inj_exec = NullerProcess('tag', processing_args, work_dir, work_dir, mode='INIT')
inj_exec.injector_setup()

70.29 53.20161863972944
22.28 23.271496418964336
23.08 24.087866786551462
51.57 41.777944928217124
27.64 34.13878713060977
21.01 27.15556762534496


AttributeError: 'NullerProcess' object has no attribute 'SNR_record'